In [65]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [66]:
 #1. Load Dataset

#file_path = r"E:\syntecxhub_project3\covid19\country_wise_latest.csv"
# OR
file_path = "E:\\syntecxhub_project3\\covid19\\country_wise_latest.csv"

def load_data(file_path):
    df = pd.read_csv(file_path)
    # Only keep this if your CSV actually has a 'Date' column
    # df['Date'] = pd.to_datetime(df['Date'])
    return df


def load_data(file_path):
    df = pd.read_csv(file_path)
    df['Date'] = pd.to_datetime(df['Date'])
    return df

In [67]:
 #2. Prepare Country Data
# ----------------------------

def prepare_country_data(df, country):
    country_df = df[df['Country'] == country].copy()
    country_df = country_df.sort_values('Date')
    
    # Daily new cases
    country_df['Daily_Cases'] = country_df['Confirmed'].diff()
    
    # Weekly cases using resample
    country_df.set_index('Date', inplace=True)
    country_df['Weekly_Cases'] = country_df['Daily_Cases'].resample('W').transform('sum')
    
    # 7-day rolling mean
    country_df['Rolling_7'] = country_df['Daily_Cases'].rolling(7).mean()
    
    return country_df


In [68]:
# 3. Compute Approx R_t
# ----------------------------

def compute_rt(df):
    # Simple proxy: cases today / cases 7 days ago
    df['Rt_Proxy'] = df['Rolling_7'] / df['Rolling_7'].shift(7)
    return df



In [69]:
# 4. Detect Peaks
# ----------------------------

def detect_peaks(df, country):
    peak_value = df['Daily_Cases'].max()
    peak_date = df['Daily_Cases'].idxmax()
    print(f"{country} Peak Cases: {int(peak_value)} on {peak_date.date()}")


In [70]:
# 5. Plot Comparison
# ----------------------------

def plot_countries(data_dict):
    plt.figure()
    
    for country, df in data_dict.items():
        plt.plot(df.index, df['Rolling_7'], label=country)
    
    plt.title("7-Day Rolling Average Comparison")
    plt.xlabel("Date")
    plt.ylabel("Daily Cases (Smoothed)")
    plt.legend()
    plt.savefig("rolling_avg_comparison.png")
    plt.show()


In [74]:
# 6. Export Summary
# ----------------------------
"""
def export_summary(data_dict):
    summary_list = []
    
    for country, df in data_dict.items():
        summary_list.append({
            "Country": country,
            "Total Cases": df['Confirmed'].iloc[-1],
            "Peak Daily Cases": df['Daily_Cases'].max(),
            "Latest Rt Proxy": round(df['Rt_Proxy'].iloc[-1], 2)
        })
    
    summary_df = pd.DataFrame(summary_list)
    summary_df.to_csv("covid_summary_report.csv", index=False)"""

import os

def export_summary(data_dict):
    summary_list = []
    
    for country, df in data_dict.items():
        summary_list.append({
            "Country": country,
            "Total Cases": df['Confirmed'].iloc[-1],
            "Peak Daily Cases": df['Daily_Cases'].max(),
            "Latest Rt Proxy": round(df['Rt_Proxy'].iloc[-1], 2)
        })
    
    summary_df = pd.DataFrame(summary_list)
    
    # Debug check
    print(summary_df.head())
    print("Saving to:", os.getcwd())
    
    # Save file
    summary_df.to_csv("covid_summary_report.csv", index=False)
    print("\nFile saved successfully.")


In [75]:
#import pandas as pd

def load_data(file_path):
    df = pd.read_csv(file_path)
    return df

def main():
    file_path = r"E:\syntecxhub_project3\covid19\country_wise_latest.csv"
    df = load_data(file_path)
    print(df.head())
    print("\nAnalysis completed successfully.")

if __name__ == "__main__":
    main()


  Country/Region  Confirmed  Deaths  Recovered  Active  New cases  New deaths  \
0    Afghanistan      36263    1269      25198    9796        106          10   
1        Albania       4880     144       2745    1991        117           6   
2        Algeria      27973    1163      18837    7973        616           8   
3        Andorra        907      52        803      52         10           0   
4         Angola        950      41        242     667         18           1   

   New recovered  Deaths / 100 Cases  Recovered / 100 Cases  \
0             18                3.50                  69.49   
1             63                2.95                  56.25   
2            749                4.16                  67.34   
3              0                5.73                  88.53   
4              0                4.32                  25.47   

   Deaths / 100 Recovered  Confirmed last week  1 week change  \
0                    5.04                35526            737   
1   